# 00b — Comparación CSV vs XLSX · P8-Data-Analyst

**Objetivo:** Determinar si los archivos `.xlsx` aportan datos nuevos respecto a los `.csv` de la misma ciudad,
o si son actualizaciones parciales/completas del mismo dataset.

**Hipótesis:** Los datasets de Inside AirBnB son snapshots temporales. Un xlsx más reciente que su CSV
equivalente contendrá registros nuevos (alojamientos dados de alta), habrá perdido registros antiguos
(dados de baja) y puede haber registros comunes con valores actualizados (precio, disponibilidad).

**Ciudades a comparar:**
- Madrid: `madrid_airbnb.csv` (2021-05) vs `madrid_airbnb.xlsx` (2026-01)
- Londres: `london_airbnb.csv` (2022-08) vs `london_airbnb.xlsx` (2024-10-24) vs `london_airbnb(1).xlsx` (2024-10-11)
- Milán: `milan_airbnb.csv` (2021-08) vs `milan_airbnb.xlsx` (2024-10)

**Ciudades sin xlsx:** NY, Sydney, Tokyo → solo existe una versión.

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

BASE = 'https://raw.githubusercontent.com/Jose-JulioRamirezySanchez-Escobar/P8-Data-Analyst/main/data/raw/'

def load_csv(nombre):
    return pd.read_csv(BASE + nombre, low_memory=False)

def load_xlsx(nombre):
    r = requests.get(BASE + nombre)
    r.raise_for_status()
    return pd.read_excel(BytesIO(r.content), engine='openpyxl')

print('✅ Funciones de carga listas')

In [ ]:
# SDD: Cargar todos los datasets con ciudad y versión etiquetadas
print('Cargando datasets...')

datasets = {
    # (ciudad, version, formato, fecha_snapshot)
    'madrid_csv':    {'df': load_csv('madrid_airbnb.csv'),     'ciudad': 'Madrid',  'fecha': '2021-05'},
    'madrid_xlsx':   {'df': load_xlsx('madrid_airbnb.xlsx'),   'ciudad': 'Madrid',  'fecha': '2026-01'},
    'london_csv':    {'df': load_csv('london_airbnb.csv'),     'ciudad': 'London',  'fecha': '2022-08'},
    'london_xlsx':   {'df': load_xlsx('london_airbnb.xlsx'),   'ciudad': 'London',  'fecha': '2024-10-24'},
    'london_xlsx1':  {'df': load_xlsx('london_airbnb(1).xlsx'),'ciudad': 'London',  'fecha': '2024-10-11'},
    'milan_csv':     {'df': load_csv('milan_airbnb.csv'),      'ciudad': 'Milan',   'fecha': '2021-08'},
    'milan_xlsx':    {'df': load_xlsx('milan_airbnb.xlsx'),    'ciudad': 'Milan',   'fecha': '2024-10'},
    # Solo CSV — sin comparación xlsx
    'ny_csv':        {'df': load_csv('NY_airbnb.csv'),         'ciudad': 'New York','fecha': '2019-10'},
    'sydney_csv':    {'df': load_csv('sydney_airbnb.csv'),     'ciudad': 'Sydney',  'fecha': '2019-11'},
    'tokyo_csv':     {'df': load_csv('tokyo_airbnb.csv'),      'ciudad': 'Tokyo',   'fecha': '2019-09'},
}

for key, meta in datasets.items():
    df = meta['df']
    print(f"  {key:15} → {df.shape[0]:>6,} filas × {df.shape[1]:>2} cols | "
          f"cols: {list(df.columns)}")

## 1. Estructura — columnas por archivo

Antes de comparar registros, comparamos estructura. Si las columnas difieren entre CSV y XLSX,
la versión más reciente puede haber añadido o eliminado campos.

In [ ]:
# SDD: Comparación de columnas CSV vs XLSX por ciudad
pares = [
    ('madrid_csv',   'madrid_xlsx',  'Madrid'),
    ('london_csv',   'london_xlsx',  'London CSV vs XLSX'),
    ('london_xlsx1', 'london_xlsx',  'London XLSX(1) vs XLSX'),
    ('milan_csv',    'milan_xlsx',   'Milan'),
]

print('=== COMPARACIÓN DE COLUMNAS ===\n')
for key_a, key_b, nombre in pares:
    cols_a = set(datasets[key_a]['df'].columns)
    cols_b = set(datasets[key_b]['df'].columns)
    solo_a = cols_a - cols_b
    solo_b = cols_b - cols_a
    comunes = cols_a & cols_b
    fecha_a = datasets[key_a]['fecha']
    fecha_b = datasets[key_b]['fecha']
    print(f'--- {nombre} ---')
    print(f'  {key_a} ({fecha_a}): {len(cols_a)} columnas')
    print(f'  {key_b} ({fecha_b}): {len(cols_b)} columnas')
    print(f'  Comunes: {len(comunes)}')
    if solo_a: print(f'  Solo en {key_a}: {solo_a}')
    if solo_b: print(f'  Solo en {key_b}: {solo_b}')
    print()

## 2. Solapamiento de IDs — ¿cuántos alojamientos persisten entre versiones?

El campo `id` identifica unívocamente cada alojamiento. Comparando los sets de IDs entre versiones
obtenemos:
- **IDs solo en el CSV antiguo:** alojamientos que desaparecieron del mercado
- **IDs solo en el XLSX nuevo:** alojamientos nuevos dados de alta
- **IDs en ambos:** alojamientos que persisten — pueden tener valores actualizados

In [ ]:
# SDD: Solapamiento de IDs por ciudad
print('=== SOLAPAMIENTO DE IDs ===\n')

resultados_ids = {}
for key_a, key_b, nombre in pares:
    df_a = datasets[key_a]['df']
    df_b = datasets[key_b]['df']
    fecha_a = datasets[key_a]['fecha']
    fecha_b = datasets[key_b]['fecha']

    if 'id' not in df_a.columns or 'id' not in df_b.columns:
        print(f'  {nombre}: columna id no encontrada')
        continue

    ids_a = set(df_a['id'].astype(str))
    ids_b = set(df_b['id'].astype(str))
    solo_a = ids_a - ids_b    # desaparecidos
    solo_b = ids_b - ids_a    # nuevos
    comunes = ids_a & ids_b   # persisten

    resultados_ids[nombre] = {
        'total_a': len(ids_a), 'total_b': len(ids_b),
        'nuevos': len(solo_b), 'desaparecidos': len(solo_a),
        'comunes': len(comunes), 'ids_comunes': comunes
    }

    pct_nuevos = len(solo_b)/len(ids_b)*100
    pct_desap  = len(solo_a)/len(ids_a)*100
    pct_comun  = len(comunes)/max(len(ids_a),len(ids_b))*100

    print(f'--- {nombre} ---')
    print(f'  {key_a} ({fecha_a}): {len(ids_a):,} IDs únicos')
    print(f'  {key_b} ({fecha_b}): {len(ids_b):,} IDs únicos')
    print(f'  IDs comunes (persisten):    {len(comunes):>6,} ({pct_comun:.1f}%)')
    print(f'  IDs solo en CSV (desapar.): {len(solo_a):>6,} ({pct_desap:.1f}%)')
    print(f'  IDs solo en XLSX (nuevos):  {len(solo_b):>6,} ({pct_nuevos:.1f}%)')
    print()

## 3. Cambios en valores para IDs comunes

Para los alojamientos que existen en ambas versiones, ¿han cambiado sus valores?
Las columnas más relevantes para detectar cambios son: `price`, `minimum_nights`,
`availability_365`, `room_type`.

In [ ]:
# SDD: Análisis de cambios en valores para IDs comunes
COLS_SEGUIMIENTO = ['price', 'minimum_nights', 'room_type']
# availability_365 solo si existe en ambos

print('=== CAMBIOS EN VALORES PARA IDs COMUNES ===\n')

for key_a, key_b, nombre in pares:
    if nombre not in resultados_ids:
        continue
    res = resultados_ids[nombre]
    if res['comunes'] == 0:
        print(f'{nombre}: sin IDs comunes — no hay cambios que analizar\n')
        continue

    df_a = datasets[key_a]['df'].copy()
    df_b = datasets[key_b]['df'].copy()
    df_a['id'] = df_a['id'].astype(str)
    df_b['id'] = df_b['id'].astype(str)

    ids_c = res['ids_comunes']
    sub_a = df_a[df_a['id'].isin(ids_c)].set_index('id')
    sub_b = df_b[df_b['id'].isin(ids_c)].set_index('id')

    print(f'--- {nombre} ({res["comunes"]:,} IDs comunes) ---')
    cols_check = [c for c in COLS_SEGUIMIENTO if c in sub_a.columns and c in sub_b.columns]
    if 'availability_365' in sub_a.columns and 'availability_365' in sub_b.columns:
        cols_check.append('availability_365')

    for col in cols_check:
        try:
            diff = (sub_a[col] != sub_b[col]).sum()
            pct = diff / res['comunes'] * 100
            print(f'  {col:35}: {diff:>6,} cambios ({pct:.1f}%)')
        except Exception as e:
            print(f'  {col}: no comparable — {e}')
    print()

## 4. Resumen ejecutivo — ¿qué aporta cada XLSX?

Interpretación de los resultados para tomar decisiones sobre qué versión usar.

In [ ]:
# SDD: Resumen ejecutivo para el catálogo de datasets
print('=== RESUMEN EJECUTIVO ===\n')
print('Criterio de interpretación:')
print('  >80% IDs nuevos  → XLSX es una actualización mayor, aporta datos muy distintos')
print('  30-80% IDs nuevos → XLSX es una actualización parcial')
print('  <30% IDs nuevos  → XLSX es principalmente el mismo dataset con pequeños cambios')
print()

for nombre, res in resultados_ids.items():
    pct_nuevos = res['nuevos'] / res['total_b'] * 100 if res['total_b'] > 0 else 0
    if pct_nuevos > 80:
        tipo = '🔴 Actualización MAYOR — datasets muy distintos'
    elif pct_nuevos > 30:
        tipo = '🟡 Actualización PARCIAL — mezcla de datos nuevos y existentes'
    else:
        tipo = '🟢 Actualización MENOR — mayoría de datos coinciden'

    print(f'{nombre}')
    print(f'  {tipo}')
    print(f'  Nuevos: {res["nuevos"]:,} | Desaparecidos: {res["desaparecidos"]:,} | Comunes: {res["comunes"]:,}')
    print(f'  Recomendación: ', end='')
    if pct_nuevos > 50:
        print('usar XLSX como versión activa — aporta datos significativamente más recientes')
    else:
        print('evaluar si los cambios justifican migrar al XLSX')
    print()

In [ ]:
# SDD: Tabla resumen para el catálogo de datasets (markdown)
import os
os.makedirs('data/processed', exist_ok=True)

lineas = ['# Comparación CSV vs XLSX por ciudad\n\n']
lineas.append('| Ciudad | Versión A | Versión B | IDs comunes | Nuevos en B | Desapar. de A | Significancia |\n')
lineas.append('|---|---|---|---|---|---|---|\n')

for (key_a, key_b, nombre), (_, res) in zip(pares, resultados_ids.items()):
    pct = res['nuevos']/res['total_b']*100 if res['total_b'] > 0 else 0
    sig = 'Alta' if pct > 50 else ('Media' if pct > 20 else 'Baja')
    lineas.append(
        f"| {nombre} | {datasets[key_a]['fecha']} ({res['total_a']:,}) "
        f"| {datasets[key_b]['fecha']} ({res['total_b']:,}) "
        f"| {res['comunes']:,} | {res['nuevos']:,} ({pct:.0f}%) "
        f"| {res['desaparecidos']:,} | {sig} |\n"
    )

ruta = 'data/processed/comparacion_csv_xlsx.md'
with open(ruta, 'w', encoding='utf-8') as f:
    f.writelines(lineas)

print(''.join(lineas))
print(f'✅ Guardado en: {ruta}')

## 5. Estructura completa de cada dataset (para el catálogo)

Output directo para completar `catalogo_datasets_data_dictionary.md`.

In [ ]:
# SDD: Ficha completa de cada dataset para el catálogo
print('=== FICHA DE CADA DATASET ===\n')

for key, meta in datasets.items():
    df = meta['df']
    print(f'--- {key} ({meta["ciudad"]} · {meta["fecha"]}) ---')
    print(f'  Registros: {df.shape[0]:,} | Columnas: {df.shape[1]}')
    print(f'  Nulos totales: {df.isnull().sum().sum():,}')
    print(f'  Columnas y tipos:')
    for col in df.columns:
        n_nulos = df[col].isnull().sum()
        pct_nulos = n_nulos / len(df) * 100
        dtype = str(df[col].dtype)
        n_uniq = df[col].nunique()
        nulos_str = f' [{n_nulos:,} nulos = {pct_nulos:.1f}%]' if n_nulos > 0 else ''
        print(f'    {col:40} {dtype:10} {n_uniq:>6} valores únicos{nulos_str}')
    print()